### RAG v2


In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('chatbot_rag_v2') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

25/04/18 03:25:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [ ]:
# !pip install langchain
# !pip install langchain_community
# !pip install qdrant-client
# !pip install python-dotenv
# !pip install -qU langchain-ollama

In [2]:
from dotenv import load_dotenv
import os

from pyspark.sql.types import StructType
from pyspark.sql import DataFrame
from pyspark.sql.functions import explode, col

In [3]:
%run ./01_Config_env.ipynb

In [4]:
# Variaveis de Ambiente
OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
token = os.getenv("API_TOKEN")

In [5]:
 %run ./02_Common.ipynb

In [6]:
%run ./03_Get_data.ipynb

Dados disponiveis:
root
 |-- c: string (nullable = true)
 |-- cl: string (nullable = true)
 |-- sl: string (nullable = true)
 |-- lt0: string (nullable = true)
 |-- lt1: string (nullable = true)
 |-- qv: string (nullable = true)
 |-- vs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- p: string (nullable = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- ta: string (nullable = true)
 |    |    |-- py: string (nullable = true)
 |    |    |-- px: string (nullable = true)
 |    |    |-- sv: string (nullable = true)
 |    |    |-- is: string (nullable = true)



### Visualizar e pegar uma amostra dos dados

In [7]:
df = df_posicao.select(
    col('c').alias('Letreiro_Linha'),
    col('cl').alias('Linha'),
    col('sl').alias('Sentido'),
    col('lt0').alias('Destino_Linha'),
    col('lt1').alias('Origem_Linha'),
    col('qv').cast('int').alias('Quantidade_Veiculos')
    
).limit(10)

df.createOrReplaceTempView("tbl_bus_posicao")
spark.sql("SELECT * FROM tbl_bus_posicao").show()

+--------------+-----+-------+--------------------+--------------------+-------------------+
|Letreiro_Linha|Linha|Sentido|       Destino_Linha|        Origem_Linha|Quantidade_Veiculos|
+--------------+-----+-------+--------------------+--------------------+-------------------+
|       1788-10|  653|      1|       METRÔ SANTANA|        JD. FONTÁLIS|                  1|
|       7004-10| 1992|      1|EST. STO. AMARO/G...|    TERM. JD. JACIRA|                 13|
|       576C-10|34058|      2|    TERM. STO. AMARO|     METRÔ JABAQUARA|                  3|
|       848L-10|33576|      2|      TERM. PIRITUBA|RECANTO DOS HUMILDES|                  2|
|       5614-10|34283|      2|    PÇA. JOÃO MENDES|            ELDORADO|                  4|
|       8528-10|33343|      2|     PÇA. DO CORREIO|         JD. GUARANI|                  1|
|       1036-10| 2453|      1|   CONEXÃO VL. IÓRIO|   COHAB BRASILÂNDIA|                  2|
|       2079-10|33138|      2| CPTM ITAIM PAULISTA|           JD. NÉLI

### Chatbot com RAG

In [14]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Qdrant
from langchain_qdrant import Qdrant
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain.prompts.chat import ChatPromptTemplate
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
import uuid


## Funções Auxiliares

In [9]:
# Remover formatação do código gerado
import re

def limpar_sql(resposta_modelo):
    # Remove blocos de código markdown e espaços extras
    sql = re.sub(r"```sql|```", "", resposta_modelo, flags=re.IGNORECASE).strip()
    return sql

In [10]:
def get_metadata(table_name="tbl_bus_posicao"):
    df = spark.sql(f"SELECT * FROM {table_name} LIMIT 1;")
    colunas = "\n".join([f"- {f.name}: {f.dataType.simpleString()}" for f in df.schema])
    return f"Tabela: {table_name}\n\nColunas:\n{colunas}"


## Qdrant Memory

In [ ]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="mistral:latest", base_url=OLLAMA_API_URL)

In [12]:
# Cria coleção para armazenar os embeddings 
# embedding = OllamaEmbeddings(model="mistral:latest", base_url=OLLAMA_API_URL)

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

qdrant_client = QdrantClient(":memory:")

qdrant_client.create_collection(
    collection_name="perguntas_sql",
    vectors_config=VectorParams(size=embedding.embed_query("iniciar").__len__(), distance=Distance.COSINE)
)

# Inicializa o banco vetorial com Langchain
qdrant_db = Qdrant(
    client=qdrant_client,
    collection_name="perguntas_sql",
    embeddings=embedding
)

retriever = qdrant_db.as_retriever(search_kwargs={"k": 5})

In [1]:
%run ./05.1_Memory.ipynb

In [16]:
qdrant_memory = QdrantMemory(qdrant_client, embedding)

## Iniciar Mistral 7B

In [34]:
llm = ChatOllama(
    model="mistral:latest", 
    base_url=OLLAMA_API_URL,
    temperature = 0
) 

### Configurar Promps: Roles System e Human

In [35]:
# Prompt para gerar SQL (Spark)
prompt_sql = ChatPromptTemplate.from_messages([
    ("system", "Você é um especialista em dados. Gere apenas a consulta SQL."),
    ("human", "Contexto:\n{context}\n Estrutura da tabela:\n{schema}\n Pergunta:\n{pergunta}"
     "Escreva uma consulta SQL (somente a SQL) para responder:\n{pergunta}")
])


# Prompt para retornar resultado ao usuario
prompt_resposta = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de dados que responde perguntas com base em resultados SQL."),
    ("human", "Pergunta:\n {pergunta} \n Resultado da consulta:\n {resultado}"
    "Gere uma resposta clara e amigável para o usuário com resultado da consulta SQL.")
])

In [36]:
# Função para gerar resposta com RAG (Tabela + Qdrant)
def resposta_com_rag(pergunta, table_name):
    print("\n💬 Pergunta recebida:", pergunta)

    #Obter metadados da tabela
    schema_txt = get_metadata(table_name)

    # Busca embeddings no Qdrant (Memoria)
    docs = retriever.invoke(pergunta)
    contexto = "\n".join([doc.page_content for doc in docs])

    # Gerar o SQL da query com base na pergunta 
    sql_result = llm.invoke(prompt_sql.format_messages(pergunta=pergunta, schema=schema_txt, context=contexto)).content.strip()
    sql_query=limpar_sql(sql_result)
    print("\n🤖💡 SQL Gerada:", sql_query)

    # Executa query no Spark
    try:
        resultado_df = spark.sql(sql_query)      
        resultado_dict = resultado_df.toPandas().to_dict(orient="records")
    except Exception as e:
        print("\n❌ Erro na execução da SQL:", e)
        return

    # Gera resposta amigável para retornar ao usuario
    resposta = llm.invoke(prompt_resposta.format_messages(pergunta=pergunta, resultado=resultado_dict)).content.strip()
    print("\n🤖 Resposta final:", resposta)

    # Armazena embeddings da pergunta + SQL no Qdrant
    qdrant_memory.ensinar(pergunta, sql_query, metadados={"tabela": table_name, "tipo": "AGG", "schema": schema_txt, "score": 1})
    

    return resposta

In [37]:
resp = resposta_com_rag("mostre todos os letreiros que começam com 8 e a soma total dos veiculos?", "tbl_bus_posicao")


💬 Pergunta recebida: mostre todos os letreiros que começam com 8 e a soma total dos veiculos?

🤖💡 SQL Gerada: SELECT Letreiro_Linha, SUM(Quantidade_Veiculos) as Total_Veiculos
   FROM tbl_bus_posicao
   WHERE Letreiro_Linha LIKE '8%'
   GROUP BY Letreiro_Linha;

🤖 Resposta final: Ótimo! A consulta mostrou dois letreiros que começam com o número 8: '848L-10' e '8528-10'. Além disso, a soma total dos veículos associados a esses letreiros é de 3 (2 veículos para '848L-10' e 1 veículo para '8528-10).

⚠️ Embedding similar já existe. Incrementando score...
Score:  6


### Listar Exemplos de Consultas

In [31]:
qdrant_memory.listar_exemplos()


🔹 Embedding 1:
Pergunta: Qual a origem da linha com letreiro 407H-10?
SQL: SELECT Origem_Linha FROM tbl_bus_posicao WHERE Letreiro_Linha = '407H-10';


In [30]:
# table_name = "tbl_bus_posicao"
# schema_txt = get_metadata(table_name)
# sql_query ="""SELECT SUM(Quantidade_Veiculos) FROM tbl_bus_posicao;"""
# qdrant_memory.ensinar(
#     "Some o total de veiculos?",
#     sql_query, 
#     metadados={"tabela": table_name, "tipo": "AGG", "schema": schema_txt, "score": 1})


⚠️ Embedding similar já existe. Incrementando score...
Score:  3
